# Setup

In [ ]:
import sys
from pathlib import Path
import os
import importlib

# os.environ["MX_DEX_ENV"] = "shadowfork4"

sys.path.append(str(Path.cwd().parent.parent.parent.absolute()))
import config
import argparse

importlib.reload(config)
from context import Context
from utils.utils_chain import WrapperAddress
from contracts.router_contract import RouterContract
from tools.runners.pair_runner import (
    upgrade_pair_contracts,
    fetch_and_save_pairs_from_chain,
    pause_pair_contracts,
    resume_pair_contracts,
)
from tools.runners.router_runner import (
    upgrade_router_contract,
    upgrade_template_pair_contract,
    resume as resume_router,
    pause as pause_router,
)
from tools.runner import fetch_and_save_pause_state

context = Context()

router_wasm = (
    "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.5.0-rc7/router.wasm"
)
router_code_hash = "091314188dfc6fefd21c1392ff32d6b8d49ed61976b8d633bf0d69edd34e412b"
pair_wasm = "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.5.0-rc7/pair.wasm"
pair_code_hash = "3d3ee01d61bd9fa94698b8e4d8da4736c6805ff9fd3ff70bfb13e3963302bd79"
pair_view_wasm = (
    "https://github.com/multiversx/mx-exchange-sc/releases/download/v3.5.0-rc7/safe-price-view.wasm"
)
pair_view_code_hash = "dd73ff57d0e2041b4a8ee841b96e450754b0bfe0ae58d7c9fe8c8f8c7b2f5a10"

Check owner balance for required minimum

In [ ]:
current_balance = context.network_provider.proxy.get_account(
    context.deployer_account.address
).balance
assert current_balance > 20 * 10**18, "Deployer account doesn't have enough balance"

Clean outputs folder

In [ ]:
import shutil

if config.UPGRADER_OUTPUT_FOLDER.exists():
    shutil.rmtree(config.UPGRADER_OUTPUT_FOLDER)

Prep contract pause states

In [ ]:
fetch_and_save_pairs_from_chain("")

In [ ]:
fetch_and_save_pause_state("")

Pause contracts

In [ ]:
pause_router("")

Upgrade router contract

In [ ]:
router_address = context.get_contracts(config.ROUTER_V2)[0].address
args = argparse.Namespace(address=router_address, bytecode=router_wasm, compare_states=True)
upgrade_router_contract(args)

code_hash = context.network_provider.proxy.get_account(
    WrapperAddress(router_address)
).contract_code_hash.hex()
assert code_hash == router_code_hash, f"Code hash mismatch for {router_address}"
print("Done!")

In [ ]:
pause_pair_contracts("")

# Safe Price upgrade procedure

Upgrade template pair contract

In [ ]:
args = argparse.Namespace(bytecode=pair_wasm, compare_states=True)
upgrade_template_pair_contract(args)

router_contract: RouterContract = context.get_contracts(config.ROUTER_V2)[0]
template_pair_address = router_contract.get_pair_template_address(context.network_provider.proxy)
code_hash = context.network_provider.proxy.get_account(
    WrapperAddress(template_pair_address)
).contract_code_hash.hex()
assert code_hash == pair_code_hash, f"Code hash mismatch for {template_pair_address}"
print("Done!")

Resume router contract

In [ ]:
resume_router("")

Upgrade pairs

In [ ]:
# TODO: update it to True
args = argparse.Namespace(compare_states=True)
upgrade_pair_contracts(args)

Upgrade view contract

In [ ]:
pair_view_contract = context.get_contracts(config.PAIRS_VIEW)[0]
context.deployer_account.sync_nonce(context.network_provider.proxy)

pair_view_contract.contract_upgrade(
    context.deployer_account, context.network_provider.proxy, pair_view_wasm, [], no_init=True
)
code_hash = context.network_provider.proxy.get_account(
    WrapperAddress(pair_view_contract.address)
).contract_code_hash.hex()
assert code_hash == pair_view_code_hash, f"Code hash mismatch for {pair_view_contract.address}"

# Resume contracts

In [ ]:
resume_pair_contracts("")